In [0]:
%pip install databricks-langchain langgraph langchain mlflow
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# LangGraph Agent with Databricks LLM
from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import mlflow

# Connect to Databricks LLM - no API key needed!
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1
)

print("Databricks LLM connected!")
response = llm.invoke("Say hello!")
print(f"Response: {response.content}")

Databricks LLM connected!
Response: Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?


In [0]:
# Text-to-SQL Tool for LangGraph Agent


from langchain_core.tools import tool

# Table schema for LLM to understand
SCHEMA = """
Bronze Delta Tables:

1. bronze_transactions
   - transactionID, customerID, franchiseID
   - dateTime, product, quantity
   - unitPrice, totalPrice, paymentMethod

2. bronze_customers
   - customerID, first_name, last_name
   - email_address, city, state
   - country, continent, gender

3. bronze_franchises
   - franchiseID, franchiseName
   - city, state, country

4. bronze_audit_log
   - source_name, status
   - rows_ingested, start_time, error_message
"""

@tool
def text_to_sql(question: str) -> str:
    """Use this tool for ANY question about sales, customers, franchises or ingestion audit data. 
    This tool generates SQL from natural language and runs it on Bronze Delta tables."""
    
    # Step 1 - Generate SQL using LLM
    sql_prompt = f"""You are a Spark SQL expert.

Given these tables:
{SCHEMA}

Generate a valid Spark SQL query to answer:
{question}

Rules:
- Return ONLY the SQL query, nothing else
- No explanations, no markdown, no backticks
- Use proper Spark SQL syntax
- Add LIMIT 20 unless question asks for specific count
"""
    
    sql_response = llm.invoke(sql_prompt)
    sql = sql_response.content.strip()
    print(f"Generated SQL: {sql}")
    
    # Step 2 - Run SQL on Bronze Delta tables
    try:
        df = spark.sql(sql)
        result = df.toPandas().to_string()
        return f"Query Result:\n{result}"
    except Exception as e:
        return f"SQL Error: {str(e)}"

print("Text-to-SQL tool created!")

Text-to-SQL tool created!


In [0]:

# LangGraph Agent + MLflow LLMOps
import mlflow
import time

mlflow.set_experiment("/Users/theepankumar72@gmail.com/langgraph_agent_llmops")
mlflow.langchain.autolog()

# Create LangGraph agent with Text-to-SQL tool
tools = [text_to_sql]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""You are a helpful data analyst for a bakehouse business. 
You have access to Bronze Delta tables via the text_to_sql tool.
Always use the text_to_sql tool to fetch data before answering any question.
Give clear and concise answers based on the data returned."""
)

print("LangGraph + Databricks LLM Agent created!")

@mlflow.trace
def ask_langgraph_agent(question):
    print(f"\nQuestion: {question}")
    print("-" * 50)
    start_time = time.time()
    
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    duration = time.time() - start_time
    answer = response["messages"][-1].content
    
    print(f"Answer: {answer}")
    print(f"Response time: {round(duration, 2)}s")
    return answer

# Test with real questions!
ask_langgraph_agent("What is the total revenue from all sales?")
ask_langgraph_agent("Which continent has the most customers?")
ask_langgraph_agent("How many franchises do we have?")

/home/spark-a0894f94-bfa1-4674-91a9-13/.ipykernel/18020/command-8476672931950070-3143339400:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


LangGraph + Databricks LLM Agent created!

Question: What is the total revenue from all sales?
--------------------------------------------------
Generated SQL: SELECT SUM(totalPrice) AS total_revenue FROM bronze_transactions LIMIT 20
Answer: The total revenue from all sales is $66,471.
Response time: 12.45s

Question: Which continent has the most customers?
--------------------------------------------------
Generated SQL: SELECT continent, COUNT(customerID) as customer_count FROM bronze_customers GROUP BY continent ORDER BY customer_count DESC LIMIT 20
Answer: The continent with the most customers is Oceania, with 108 customers.
Response time: 2.77s

Question: How many franchises do we have?
--------------------------------------------------
Generated SQL: SELECT COUNT(DISTINCT franchiseID) FROM bronze_franchises LIMIT 20
Answer: We have 48 franchises.
Response time: 2.69s


'We have 48 franchises.'

[Trace(trace_id=tr-8aa19911089cb4fe0c4a7829c6e29af6), Trace(trace_id=tr-01acd706b9fdeb1580f8d4af951ac82c), Trace(trace_id=tr-6b6a68b9993bce4b86a7ace632ba131d)]